In [ ]:
!pip install -q openpyxl

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RAW_DIR = Path("/content/drive/MyDrive/PRT564_Merge_Project/raw_data")
OUT_DIR = Path("/content/drive/MyDrive/PRT564_Merge_Project/output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR =", RAW_DIR)
print("OUT_DIR =", OUT_DIR)

Mounted at /content/drive
RAW_DIR = /content/drive/MyDrive/PRT564_Merge_Project/raw_data
OUT_DIR = /content/drive/MyDrive/PRT564_Merge_Project/output


In [ ]:
required_files = [
    "850109.xlsx",
    "6202012.xlsx",
    "6401017.xlsx",
    "f01hist.xlsx",
    "634501.xlsx",
    "5206025_SFD_Summary.xlsx",
    "31010do001_202506.xlsx",
]

missing = [f for f in required_files if not (RAW_DIR / f).exists()]

if missing:
    print("Missing files:")
    for f in missing:
        print("-", f)
else:
    print("All required files are present.")

All required files are present.


In [ ]:
def norm_text(x) -> str:
    return " ".join(str(x).replace("\n", " ").split()).strip()


def abs_timeseries_raw(path: Path, sheet_name: str = "Data1") -> pd.DataFrame:
    return pd.read_excel(path, sheet_name=sheet_name, header=None)


def extract_abs_series(path: Path, sheet_name: str, desc_target: str, series_type_target: str, value_name: str) -> pd.DataFrame:
    raw = abs_timeseries_raw(path, sheet_name)
    desc_target_n = norm_text(desc_target)
    series_type_target_n = norm_text(series_type_target)

    match_col = None
    for col in range(1, raw.shape[1]):
        desc = raw.iat[0, col]
        series_type = raw.iat[2, col]
        if norm_text(desc) == desc_target_n and norm_text(series_type) == series_type_target_n:
            match_col = col
            break

    if match_col is None:
        raise ValueError(f"Series not found: {desc_target} [{series_type_target}] in {path.name}")

    out = pd.DataFrame({
        "Date": pd.to_datetime(raw.iloc[10:, 0], errors="coerce"),
        value_name: pd.to_numeric(raw.iloc[10:, match_col], errors="coerce"),
    }).dropna(subset=["Date"])

    return out


def add_qoq_by_group(df: pd.DataFrame, group_col: str, value_col: str, out_col: str) -> pd.DataFrame:
    df = df.sort_values([group_col, "Quarter_End"]).copy()
    df[out_col] = df.groupby(group_col)[value_col].pct_change() * 100
    return df


def add_qoq_single(df: pd.DataFrame, value_col: str, out_col: str) -> pd.DataFrame:
    df = df.sort_values("Quarter_End").copy()
    df[out_col] = df[value_col].pct_change() * 100
    return df

In [ ]:
def extract_turnover_state_panel() -> pd.DataFrame:
    states = {
        "New South Wales": "NSW",
        "Victoria": "VIC",
        "Queensland": "QLD",
        "South Australia": "SA",
        "Western Australia": "WA",
        "Tasmania": "TAS",
        "Northern Territory": "NT",
        "Australian Capital Territory": "ACT",
    }

    frames = []
    for state_full, state_code in states.items():
        desc = f"Turnover - Chain Volume Measure ;  {state_full} ;  Total (Industry) ;"

        df_state = extract_abs_series(
            RAW_DIR / "850109.xlsx",
            "Data1",
            desc,
            "Seasonally Adjusted",
            "Retail_Turnover_CVM_SA"
        )

        df_state["State"] = state_code
        frames.append(df_state)

    out = pd.concat(frames, ignore_index=True)
    out["Quarter"] = out["Date"].dt.to_period("Q").astype(str)
    out["Quarter_End"] = out["Date"].dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()

    return out[["Quarter", "Quarter_End", "State", "Retail_Turnover_CVM_SA"]]

turnover = extract_turnover_state_panel()
print("Turnover shape:", turnover.shape)
display(turnover.head())

Turnover shape: (1344, 4)


,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA
0,1983Q3,1983-09-30,NSW,9904.6
1,1983Q4,1983-12-31,NSW,10028.8
2,1984Q1,1984-03-31,NSW,10012.0
3,1984Q2,1984-06-30,NSW,9989.5
4,1984Q3,1984-09-30,NSW,10091.4


In [ ]:
def extract_unemployment_quarterly_6state() -> pd.DataFrame:
    states = {
        "New South Wales": "NSW",
        "Victoria": "VIC",
        "Queensland": "QLD",
        "South Australia": "SA",
        "Western Australia": "WA",
        "Tasmania": "TAS",
    }

    frames = []
    for state_full, state_code in states.items():
        desc = f"Unemployment rate ;  Persons ;  > {state_full} ;"

        monthly = extract_abs_series(
            RAW_DIR / "6202012.xlsx",
            "Data2",
            desc,
            "Seasonally Adjusted",
            "Unemployment_Rate_Monthly_SA"
        )

        monthly["State"] = state_code
        monthly["Quarter"] = monthly["Date"].dt.to_period("Q").astype(str)
        monthly["Quarter_End"] = monthly["Date"].dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()

        qtr = (
            monthly
            .groupby(["Quarter", "Quarter_End", "State"], as_index=False)["Unemployment_Rate_Monthly_SA"]
            .mean()
            .rename(columns={"Unemployment_Rate_Monthly_SA": "Unemployment_Rate_QtrAvg_SA"})
        )

        frames.append(qtr)

    return pd.concat(frames, ignore_index=True)

unemp = extract_unemployment_quarterly_6state()
print("Unemployment shape:", unemp.shape)
display(unemp.head())

Unemployment shape: (1158, 4)


,Quarter,Quarter_End,State,Unemployment_Rate_QtrAvg_SA
0,1978Q1,1978-03-31,NSW,6.644107
1,1978Q2,1978-06-30,NSW,6.344612
2,1978Q3,1978-09-30,NSW,6.211181
3,1978Q4,1978-12-31,NSW,6.099391
4,1979Q1,1979-03-31,NSW,6.036644


In [ ]:
def extract_cpi_quarterly() -> pd.DataFrame:
    df_cpi = extract_abs_series(
        RAW_DIR / "6401017.xlsx",
        "Data1",
        "Index Numbers ;  All groups CPI ;  Australia ;",
        "Original",
        "CPI_All_Groups_Index"
    )

    df_cpi["Quarter"] = df_cpi["Date"].dt.to_period("Q").astype(str)
    df_cpi["Quarter_End"] = df_cpi["Date"].dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()

    return df_cpi[["Quarter", "Quarter_End", "CPI_All_Groups_Index"]]

cpi = extract_cpi_quarterly()
print("CPI shape:", cpi.shape)
display(cpi.head())

CPI shape: (310, 3)


,Quarter,Quarter_End,CPI_All_Groups_Index
10,1948Q3,1948-09-30,2.59
11,1948Q4,1948-12-31,2.63
12,1949Q1,1949-03-31,2.71
13,1949Q2,1949-06-30,2.74
14,1949Q3,1949-09-30,2.82


In [ ]:
def extract_cash_rate_quarterly() -> pd.DataFrame:
    raw = pd.read_excel(RAW_DIR / "f01hist.xlsx", sheet_name="Data", header=None)

    df_cash = pd.DataFrame({
        "Date": pd.to_datetime(raw.iloc[11:, 0], errors="coerce"),
        "Cash_Rate_Target_Monthly_Avg": pd.to_numeric(raw.iloc[11:, 1], errors="coerce"),
    }).dropna(subset=["Date"])

    df_cash["Quarter"] = df_cash["Date"].dt.to_period("Q").astype(str)
    df_cash["Quarter_End"] = df_cash["Date"].dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()

    qtr = (
        df_cash.groupby(["Quarter", "Quarter_End"], as_index=False)["Cash_Rate_Target_Monthly_Avg"]
        .mean()
        .rename(columns={"Cash_Rate_Target_Monthly_Avg": "Cash_Rate_QtrAvg"})
    )

    return qtr

cash = extract_cash_rate_quarterly()
print("Cash rate shape:", cash.shape)
display(cash.head())

Cash rate shape: (228, 3)


,Quarter,Quarter_End,Cash_Rate_QtrAvg
0,1969Q2,1969-06-30,NaN
1,1969Q3,1969-09-30,NaN
2,1969Q4,1969-12-31,NaN
3,1970Q1,1970-03-31,NaN
4,1970Q2,1970-06-30,NaN


In [ ]:
def extract_wpi_quarterly() -> pd.DataFrame:
    df_wpi = extract_abs_series(
        RAW_DIR / "634501.xlsx",
        "Data1",
        "Quarterly Index ;  Total hourly rates of pay excluding bonuses ;  Australia ;  Private and Public ;  All industries ;",
        "Seasonally Adjusted",
        "WPI_Index_SA"
    )

    df_wpi["Quarter"] = df_wpi["Date"].dt.to_period("Q").astype(str)
    df_wpi["Quarter_End"] = df_wpi["Date"].dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()

    return df_wpi[["Quarter", "Quarter_End", "WPI_Index_SA"]]

wpi = extract_wpi_quarterly()
print("WPI shape:", wpi.shape)
display(wpi.head())

WPI shape: (114, 3)


,Quarter,Quarter_End,WPI_Index_SA
10,1997Q3,1997-09-30,66.6
11,1997Q4,1997-12-31,67.2
12,1998Q1,1998-03-31,67.8
13,1998Q2,1998-06-30,68.3
14,1998Q3,1998-09-30,68.8


In [ ]:
def extract_sfd_state_panel() -> pd.DataFrame:
    states = {
        "New South Wales": "NSW",
        "Victoria": "VIC",
        "Queensland": "QLD",
        "South Australia": "SA",
        "Western Australia": "WA",
        "Tasmania": "TAS",
        "Northern Territory": "NT",
        "Australian Capital Territory": "ACT",
    }

    frames = []
    for state_full, state_code in states.items():
        desc = f"{state_full} ;  STATE FINAL DEMAND ;"

        df_state = extract_abs_series(
            RAW_DIR / "5206025_SFD_Summary.xlsx",
            "Data1",
            desc,
            "Seasonally Adjusted",
            "State_Final_Demand_SA"
        )

        df_state["State"] = state_code
        df_state["Quarter"] = df_state["Date"].dt.to_period("Q").astype(str)
        df_state["Quarter_End"] = df_state["Date"].dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()

        frames.append(df_state[["Quarter", "Quarter_End", "State", "State_Final_Demand_SA"]])

    return pd.concat(frames, ignore_index=True)

sfd = extract_sfd_state_panel()
print("State Final Demand shape:", sfd.shape)
display(sfd.head())

State Final Demand shape: (1296, 4)


,Quarter,Quarter_End,State,State_Final_Demand_SA
0,1985Q3,1985-09-30,NSW,64918
1,1985Q4,1985-12-31,NSW,66372
2,1986Q1,1986-03-31,NSW,66018
3,1986Q2,1986-06-30,NSW,67903
4,1986Q3,1986-09-30,NSW,68205


In [ ]:
def extract_population_quarterly_recent() -> pd.DataFrame:
    raw = pd.read_excel(RAW_DIR / "31010do001_202506.xlsx", sheet_name="Table_5", header=None)

    state_cols = {
        1: "NSW",
        2: "VIC",
        3: "QLD",
        4: "SA",
        5: "WA",
        6: "TAS",
        7: "NT",
        8: "ACT",
    }

    persons_row = raw.index[raw[0].astype(str).str.strip().eq("PERSONS")]
    if len(persons_row) == 0:
        raise ValueError("PERSONS section not found.")
    persons_row = int(persons_row[0])

    quarter_month_map = {"March": 3, "June": 6, "September": 9, "December": 12}

    records = []
    current_year = None

    for i in range(persons_row + 1, raw.shape[0]):
        label = raw.iat[i, 0]

        if pd.isna(label):
            continue

        if isinstance(label, (int, float)) and not pd.isna(label):
            current_year = int(label)
            continue

        label_str = str(label).strip()

        if label_str in quarter_month_map and current_year in {2023, 2024, 2025}:
            month = quarter_month_map[label_str]
            q_end = pd.Timestamp(year=current_year, month=month, day=1) + pd.offsets.MonthEnd(0)
            q_label = pd.Period(q_end, freq="Q").strftime("%YQ%q")

            for col, state_code in state_cols.items():
                records.append({
                    "Quarter": q_label,
                    "Quarter_End": q_end.normalize(),
                    "State": state_code,
                    "Population": pd.to_numeric(raw.iat[i, col], errors="coerce"),
                })

    pop = pd.DataFrame(records).sort_values(["State", "Quarter_End"]).reset_index(drop=True)
    return pop

pop_recent = extract_population_quarterly_recent()
print("Population shape:", pop_recent.shape)
display(pop_recent.head())

Population shape: (72, 4)


,Quarter,Quarter_End,State,Population
0,2023Q2,2023-06-30,ACT,471340
1,2023Q3,2023-09-30,ACT,473817
2,2023Q4,2023-12-31,ACT,474862
3,2024Q1,2024-03-31,ACT,477529
4,2024Q2,2024-06-30,ACT,478430


In [ ]:
national = (
    cpi.merge(cash, on=["Quarter", "Quarter_End"], how="inner")
       .merge(wpi, on=["Quarter", "Quarter_End"], how="inner")
)

print("National macro table shape:", national.shape)
display(national.head())

National macro table shape: (114, 5)


,Quarter,Quarter_End,CPI_All_Groups_Index,Cash_Rate_QtrAvg,WPI_Index_SA
0,1997Q3,1997-09-30,46.28,5.152174,66.6
1,1997Q4,1997-12-31,46.39,5.000000,67.2
2,1998Q1,1998-03-31,46.51,5.000000,67.8
3,1998Q2,1998-06-30,46.78,5.000000,68.3
4,1998Q3,1998-09-30,46.91,5.000000,68.8


In [ ]:
main_states = ["NSW", "VIC", "QLD", "SA", "WA", "TAS"]

turnover_6 = turnover[turnover["State"].isin(main_states)].copy()
sfd_6 = sfd[sfd["State"].isin(main_states)].copy()
pop_recent_6 = pop_recent[pop_recent["State"].isin(main_states)].copy()

print("Turnover_6 shape:", turnover_6.shape)
print("SFD_6 shape:", sfd_6.shape)
print("Population_6 shape:", pop_recent_6.shape)

Turnover_6 shape: (1008, 4)
SFD_6 shape: (972, 4)
Population_6 shape: (54, 4)


In [ ]:
panel_no_pop = (
    turnover_6
    .merge(unemp, on=["Quarter", "Quarter_End", "State"], how="inner")
    .merge(sfd_6, on=["Quarter", "Quarter_End", "State"], how="inner")
    .merge(national, on=["Quarter", "Quarter_End"], how="inner")
)

panel_no_pop = panel_no_pop[panel_no_pop["Quarter_End"] <= pd.Timestamp("2025-06-30")].copy()

print("No-pop panel shape:", panel_no_pop.shape)
display(panel_no_pop.head())

No-pop panel shape: (672, 9)


,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA,Unemployment_Rate_QtrAvg_SA,State_Final_Demand_SA,CPI_All_Groups_Index,Cash_Rate_QtrAvg,WPI_Index_SA
0,1997Q3,1997-09-30,NSW,15648.9,7.759386,94122,46.28,5.152174,66.6
1,1997Q4,1997-12-31,NSW,15804.2,7.453965,97053,46.39,5.000000,67.2
2,1998Q1,1998-03-31,NSW,15791.8,7.257086,97387,46.51,5.000000,67.8
3,1998Q2,1998-06-30,NSW,15699.5,7.156576,97546,46.78,5.000000,68.3
4,1998Q3,1998-09-30,NSW,15654.9,7.154846,100036,46.91,5.000000,68.8


In [ ]:
panel_no_pop = add_qoq_by_group(panel_no_pop, "State", "Retail_Turnover_CVM_SA", "Retail_Turnover_QoQ_Pct")
panel_no_pop = add_qoq_by_group(panel_no_pop, "State", "Unemployment_Rate_QtrAvg_SA", "Unemployment_QoQ_Pct")
panel_no_pop = add_qoq_by_group(panel_no_pop, "State", "State_Final_Demand_SA", "State_Final_Demand_QoQ_Pct")

for col, new_col in [
    ("CPI_All_Groups_Index", "CPI_QoQ_Pct"),
    ("Cash_Rate_QtrAvg", "Cash_Rate_QoQ_Pct"),
    ("WPI_Index_SA", "WPI_QoQ_Pct"),
]:
    tmp = add_qoq_single(panel_no_pop[["Quarter", "Quarter_End", col]].drop_duplicates(), col, new_col)
    panel_no_pop = panel_no_pop.merge(tmp, on=["Quarter", "Quarter_End", col], how="left")

display(panel_no_pop.head())

,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA,Unemployment_Rate_QtrAvg_SA,State_Final_Demand_SA,CPI_All_Groups_Index,Cash_Rate_QtrAvg,WPI_Index_SA,Retail_Turnover_QoQ_Pct,Unemployment_QoQ_Pct,State_Final_Demand_QoQ_Pct,CPI_QoQ_Pct,Cash_Rate_QoQ_Pct,WPI_QoQ_Pct
0,1997Q3,1997-09-30,NSW,15648.9,7.759386,94122,46.28,5.152174,66.6,NaN,NaN,NaN,NaN,NaN,NaN
1,1997Q4,1997-12-31,NSW,15804.2,7.453965,97053,46.39,5.000000,67.2,0.992402,-3.936145,3.114043,0.237684,-2.953586,0.900901
2,1998Q1,1998-03-31,NSW,15791.8,7.257086,97387,46.51,5.000000,67.8,-0.078460,-2.641261,0.344142,0.258676,0.000000,0.892857
3,1998Q2,1998-06-30,NSW,15699.5,7.156576,97546,46.78,5.000000,68.3,-0.584481,-1.384993,0.163266,0.580520,0.000000,0.737463
4,1998Q3,1998-09-30,NSW,15654.9,7.154846,100036,46.91,5.000000,68.8,-0.284085,-0.024172,2.552642,0.277897,0.000000,0.732064


In [ ]:
panel_no_pop = panel_no_pop[
    [
        "Quarter", "Quarter_End", "State",
        "Retail_Turnover_CVM_SA", "Retail_Turnover_QoQ_Pct",
        "Unemployment_Rate_QtrAvg_SA", "Unemployment_QoQ_Pct",
        "CPI_All_Groups_Index", "CPI_QoQ_Pct",
        "Cash_Rate_QtrAvg", "Cash_Rate_QoQ_Pct",
        "WPI_Index_SA", "WPI_QoQ_Pct",
        "State_Final_Demand_SA", "State_Final_Demand_QoQ_Pct",
    ]
].sort_values(["State", "Quarter_End"]).reset_index(drop=True)

print("Final no-pop shape:", panel_no_pop.shape)
display(panel_no_pop.head())

Final no-pop shape: (672, 15)


,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA,Retail_Turnover_QoQ_Pct,Unemployment_Rate_QtrAvg_SA,Unemployment_QoQ_Pct,CPI_All_Groups_Index,CPI_QoQ_Pct,Cash_Rate_QtrAvg,Cash_Rate_QoQ_Pct,WPI_Index_SA,WPI_QoQ_Pct,State_Final_Demand_SA,State_Final_Demand_QoQ_Pct
0,1997Q3,1997-09-30,NSW,15648.9,NaN,7.759386,NaN,46.28,NaN,5.152174,NaN,66.6,NaN,94122,NaN
1,1997Q4,1997-12-31,NSW,15804.2,0.992402,7.453965,-3.936145,46.39,0.237684,5.000000,-2.953586,67.2,0.900901,97053,3.114043
2,1998Q1,1998-03-31,NSW,15791.8,-0.078460,7.257086,-2.641261,46.51,0.258676,5.000000,0.000000,67.8,0.892857,97387,0.344142
3,1998Q2,1998-06-30,NSW,15699.5,-0.584481,7.156576,-1.384993,46.78,0.580520,5.000000,0.000000,68.3,0.737463,97546,0.163266
4,1998Q3,1998-09-30,NSW,15654.9,-0.284085,7.154846,-0.024172,46.91,0.277897,5.000000,0.000000,68.8,0.732064,100036,2.552642


In [ ]:
panel_with_pop = panel_no_pop.merge(pop_recent_6, on=["Quarter", "Quarter_End", "State"], how="inner")

panel_with_pop["Retail_Turnover_Per_Capita"] = (
    panel_with_pop["Retail_Turnover_CVM_SA"] / panel_with_pop["Population"]
)

panel_with_pop = add_qoq_by_group(panel_with_pop, "State", "Population", "Population_QoQ_Pct")

panel_with_pop = panel_with_pop[
    [
        "Quarter", "Quarter_End", "State",
        "Retail_Turnover_CVM_SA", "Retail_Turnover_QoQ_Pct", "Retail_Turnover_Per_Capita",
        "Unemployment_Rate_QtrAvg_SA", "Unemployment_QoQ_Pct",
        "CPI_All_Groups_Index", "CPI_QoQ_Pct",
        "Cash_Rate_QtrAvg", "Cash_Rate_QoQ_Pct",
        "WPI_Index_SA", "WPI_QoQ_Pct",
        "State_Final_Demand_SA", "State_Final_Demand_QoQ_Pct",
        "Population", "Population_QoQ_Pct",
    ]
].sort_values(["State", "Quarter_End"]).reset_index(drop=True)

print("Final with-pop shape:", panel_with_pop.shape)
display(panel_with_pop.head())

Final with-pop shape: (54, 18)


,Quarter,Quarter_End,State,Retail_Turnover_CVM_SA,Retail_Turnover_QoQ_Pct,Retail_Turnover_Per_Capita,Unemployment_Rate_QtrAvg_SA,Unemployment_QoQ_Pct,CPI_All_Groups_Index,CPI_QoQ_Pct,Cash_Rate_QtrAvg,Cash_Rate_QoQ_Pct,WPI_Index_SA,WPI_QoQ_Pct,State_Final_Demand_SA,State_Final_Demand_QoQ_Pct,Population,Population_QoQ_Pct
0,2023Q2,2023-06-30,NSW,32885.6,-1.357306,0.003935,3.161581,-2.314518,92.87,0.857950,3.826667,15.609265,145.6,0.831025,204496,1.187054,8356387,NaN
1,2023Q3,2023-09-30,NSW,32781.8,-0.315640,0.003900,3.424284,8.309210,93.94,1.152148,4.100000,7.142857,147.5,1.304945,204684,0.091933,8406338,0.597758
2,2023Q4,2023-12-31,NSW,32708.8,-0.222685,0.003877,3.512586,2.578713,94.53,0.628060,4.246667,3.577236,149.1,1.084746,202890,-0.876473,8437334,0.368722
3,2024Q1,2024-03-31,NSW,32622.5,-0.263843,0.003848,3.856601,9.793768,95.41,0.930921,4.350000,2.433281,150.2,0.737760,204236,0.663414,8477401,0.474877
4,2024Q2,2024-06-30,NSW,32609.7,-0.039237,0.003840,3.890362,0.875403,96.41,1.048108,4.350000,0.000000,151.5,0.865513,203288,-0.464169,8492050,0.172801


In [ ]:
readme_lines = [
    ["Merged quarterly regression datasets"],
    [None],
    ["What is included"],
    ["1. State_Qtr_SA_NoPop: rigorous 6-state quarterly panel (NSW, VIC, QLD, SA, WA, TAS), 1997Q3 to 2025Q2."],
    ["2. State_Qtr_SA_WithPop: same 6-state panel with direct quarterly population control, limited to 2023Q2 to 2025Q2."],
    ["3. Source_Mapping: variable-to-file mapping used in the merge."],
    [None],
    ["Important notes"],
    ["- Dependent variable: quarterly retail turnover chain volume measure, seasonally adjusted, by state."],
    ["- NT and ACT are excluded from the main panel because the supplied labour-force workbook does not include seasonally adjusted unemployment-rate series for those two jurisdictions."],
    ["- Population control is limited to 2023Q2–2025Q2 because the supplied population workbook only contains direct quarterly PERSONS data for that window."],
    ["- CPI, Cash Rate, and WPI are national series and are repeated across states within the same quarter in the state panel."],
]
readme_df = pd.DataFrame(readme_lines)

source_map_rows = [
    ["Variable-to-source mapping for the merged quarterly datasets", None, None, None, None, None],
    [None, None, None, None, None, None],
    ["Output_Variable", "Source_File", "Source_Table", "Series_Description", "Series_Type_Used", "Notes"],
    ["Retail_Turnover_CVM_SA", "850109.xlsx", "Table 9 / Data1", "Turnover - Chain Volume Measure ; <State> ; Total (Industry) ;", "Seasonally Adjusted", "Quarterly dependent variable"],
    ["Unemployment_Rate_QtrAvg_SA", "6202012.xlsx", "Table 12 / Data2", "Unemployment rate ; Persons ; <State>", "Seasonally Adjusted monthly -> quarterly average", "6 states only (NSW, VIC, QLD, SA, WA, TAS)"],
    ["CPI_All_Groups_Index", "6401017.xlsx", "Table 17 / Data1", "Index Numbers ; All groups CPI ; Australia ;", "Original quarterly index", "National quarterly CPI"],
    ["Cash_Rate_QtrAvg", "f01hist.xlsx", "F1.1 / Data", "Cash Rate Target", "Monthly average -> quarterly average", "National quarterly average cash rate"],
    ["WPI_Index_SA", "634501.xlsx", "Table 1 / Data1", "Quarterly Index ; Total hourly rates of pay excluding bonuses ; Australia ; Private and Public ; All industries ;", "Seasonally Adjusted", "National quarterly WPI"],
    ["State_Final_Demand_SA", "5206025_SFD_Summary.xlsx", "Table 25 / Data1", "<State> ; STATE FINAL DEMAND ;", "Seasonally Adjusted", "Quarterly state final demand"],
    ["Population", "31010do001_202506.xlsx", "Table_5 PERSONS section", "Quarterly state population", "Direct quarterly PERSONS data", "Available only 2023Q2–2025Q2 in the supplied file"],
]
source_map_df = pd.DataFrame(source_map_rows)

In [ ]:
output_file = OUT_DIR / "quarterly_regression_merged_dataset.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # README
    readme_df.to_excel(writer, sheet_name="README", index=False, header=False)

    # Source mapping
    source_map_df.to_excel(writer, sheet_name="Source_Mapping", index=False, header=False)

    # Data sheets with the same top title row pattern
    title_no_pop = pd.DataFrame([["6-state quarterly merged panel without population control"]])
    title_no_pop.to_excel(writer, sheet_name="State_Qtr_SA_NoPop", index=False, header=False)
    pd.DataFrame([[None] * len(panel_no_pop.columns)]).to_excel(
        writer, sheet_name="State_Qtr_SA_NoPop", index=False, header=False, startrow=1
    )
    panel_no_pop.to_excel(writer, sheet_name="State_Qtr_SA_NoPop", index=False, startrow=2)

    title_with_pop = pd.DataFrame([["6-state quarterly merged panel with direct quarterly population control"]])
    title_with_pop.to_excel(writer, sheet_name="State_Qtr_SA_WithPop", index=False, header=False)
    pd.DataFrame([[None] * len(panel_with_pop.columns)]).to_excel(
        writer, sheet_name="State_Qtr_SA_WithPop", index=False, header=False, startrow=1
    )
    panel_with_pop.to_excel(writer, sheet_name="State_Qtr_SA_WithPop", index=False, startrow=2)

print("Created:", output_file)

Created: /content/drive/MyDrive/PRT564_Merge_Project/output/quarterly_regression_merged_dataset.xlsx
